# Weight initialisation: what not to do
Four bad ways to start the weights, each tried in Keras or numpy: all zeros, one constant for all,
small random numbers and large random numbers.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.datasets import make_moons

tf.config.experimental.enable_op_determinism()   # same numbers on every run
np.set_printoptions(precision=4, suppress=True)

## The data: 300 moons points, standardised

In [2]:
X, y = make_moons(n_samples=300, noise=0.1, random_state=42)
X = ((X - X.mean(0)) / X.std(0)).astype("float32")
print(X.shape, np.bincount(y))

(300, 2) [150 150]


## A helper: build a network, overwrite its starting weights, train it
`start(shape)` returns the starting values for an array of that shape; biases start at 0
unless `bias_too=True`.

In [3]:
def run(hidden, act, start, bias_too=False, epochs=100, seed=0, lr=0.01, sgd=False):
    keras.utils.set_random_seed(seed)
    layers = [keras.Input(shape=(2,))]
    layers += [keras.layers.Dense(n, activation=act) for n in hidden]
    layers += [keras.layers.Dense(1, activation="sigmoid")]
    m = keras.Sequential(layers)
    new = []
    for w in m.get_weights():
        if w.ndim == 2 or bias_too:
            new.append(start(w.shape).astype("float32"))
        else:
            new.append(np.zeros_like(w))
    m.set_weights(new)                           # replace Keras' default starting weights
    opt = keras.optimizers.SGD(lr) if sgd else keras.optimizers.Adam(lr)
    m.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(X, y, epochs=epochs, batch_size=32, verbose=0)
    return m, h.history

zeros = lambda s: np.zeros(s)
const = lambda s: np.full(s, 0.5)

## Case 1: all weights and biases 0, ReLU and tanh (2 hidden nodes)

In [4]:
for act in ("relu", "tanh"):
    m, h = run([2], act, zeros, bias_too=True)
    W = m.get_weights()
    print(f"--- {act}: accuracy after 100 epochs {h['accuracy'][-1]:.2f}")
    print("hidden weights:\n", W[0], "\noutput weights:", W[2].ravel(), " output bias:", W[3])

--- relu: accuracy after 100 epochs 0.47
hidden weights:
 [[0. 0.]
 [0. 0.]] 
output weights: [0. 0.]  output bias: [-0.0024]


--- tanh: accuracy after 100 epochs 0.47
hidden weights:
 [[0. 0.]
 [0. 0.]] 
output weights: [0. 0.]  output bias: [-0.0024]


## Case 1 again: all 0 with sigmoid, 10 hidden nodes
Row 1 of the weight matrix holds the 10 weights leaving input x1, row 2 those leaving x2.

In [5]:
m_sig0, h = run([10], "sigmoid", zeros, bias_too=True, epochs=200)
W = m_sig0.get_weights()
print("accuracy:", round(h["accuracy"][-1], 2))
print("weights from x1:", W[0][0])
print("weights from x2:", W[0][1])
print("hidden biases:  ", W[1])
print("distinct values per row:", [len(np.unique(r.round(6))) for r in W[0]])

accuracy: 0.87
weights from x1: [0.436 0.436 0.436 0.436 0.436 0.436 0.436 0.436 0.436 0.436]
weights from x2: [-1.1859 -1.1859 -1.1859 -1.1859 -1.1859 -1.1859 -1.1859 -1.1859 -1.1859
 -1.1859]
hidden biases:   [-0.8224 -0.8224 -0.8224 -0.8224 -0.8224 -0.8224 -0.8224 -0.8224 -0.8224
 -0.8224]
distinct values per row: [1, 1]


Same network with Keras' default (random) starting weights, for comparison.

In [6]:
keras.utils.set_random_seed(0)
m_sig_rand = keras.Sequential([keras.Input(shape=(2,)), keras.layers.Dense(10, activation="sigmoid"),
                               keras.layers.Dense(1, activation="sigmoid")])
m_sig_rand.compile(optimizer=keras.optimizers.Adam(0.01), loss="binary_crossentropy", metrics=["accuracy"])
h = m_sig_rand.fit(X, y, epochs=200, batch_size=32, verbose=0)
print("random start, accuracy:", round(h.history["accuracy"][-1], 2))

g = np.linspace(-2.5, 2.5, 151)
gx, gy = np.meshgrid(g, g)
grid = np.c_[gx.ravel(), gy.ravel()].astype("float32")
pd.DataFrame({"x1": grid[:, 0], "x2": grid[:, 1],
              "zeros": m_sig0.predict(grid, verbose=0).ravel(),
              "random": m_sig_rand.predict(grid, verbose=0).ravel()}).to_csv("data/boundary_grid.csv", index=False, float_format="%.4g")
pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y}).to_csv("data/moons.csv", index=False)

random start, accuracy: 0.96


## Case 2: every weight and bias 0.5 (3 hidden nodes)

In [7]:
for act in ("relu", "tanh", "sigmoid"):
    m, h = run([3], act, const, bias_too=True)
    W = m.get_weights()
    print(f"--- {act}: accuracy {h['accuracy'][-1]:.2f}")
    print("weights from x1:", W[0][0], "  weights from x2:", W[0][1])

--- relu: accuracy 0.88
weights from x1: [0.3115 0.3115 0.3115]   weights from x2: [-0.8416 -0.8416 -0.8416]


--- tanh: accuracy 0.88
weights from x1: [0.1919 0.1919 0.1919]   weights from x2: [-0.5444 -0.5444 -0.5444]


--- sigmoid: accuracy 0.88
weights from x1: [0.7443 0.7443 0.7443]   weights from x2: [-2.0621 -2.0621 -2.0621]


## Case 3: small random weights, 0.01 x standard normal
Four hidden layers of 10 nodes, tanh or ReLU. Plain SGD (learning rate 0.1), so every step is
learning rate x gradient; then Adam (learning rate 0.01), which rescales small gradients.

In [8]:
small = lambda s: np.random.default_rng(1).standard_normal(s) * 0.01
results = []
for opt_name, lr, sgd in (("SGD", 0.1, True), ("Adam", 0.01, False)):
    for act in ("tanh", "relu"):
        m, h = run([10, 10, 10, 10], act, small, epochs=100, lr=lr, sgd=sgd)
        w_after = m.get_weights()[0]
        print(f"{opt_name} {act}: loss {h['loss'][0]:.4f} -> {h['loss'][-1]:.4f}, accuracy {h['accuracy'][-1]:.2f}")
        print("   first-layer weights before:", small((2, 10))[0, :4], " after:", w_after[0, :4])
        results.append(dict(name=f"{act}, 0.01 x randn ({opt_name})", loss=h["loss"]))
# the same networks with Keras' default starting weights, plain SGD
for act in ("tanh", "relu"):
    keras.utils.set_random_seed(0)
    m = keras.Sequential([keras.Input(shape=(2,))] + [keras.layers.Dense(10, activation=act) for _ in range(4)]
                         + [keras.layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=keras.optimizers.SGD(0.1), loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(X, y, epochs=100, batch_size=32, verbose=0).history
    print(f"SGD {act}, Keras default start: loss {h['loss'][0]:.4f} -> {h['loss'][-1]:.4f}, "
          f"accuracy {h['accuracy'][-1]:.2f}")
    results.append(dict(name=f"{act}, Keras default (SGD)", loss=h["loss"]))
pd.DataFrame({r["name"]: r["loss"] for r in results}).rename_axis("epoch").to_csv("data/small_init_loss.csv")

SGD tanh: loss 0.6934 -> 0.6934, accuracy 0.47
   first-layer weights before: [ 0.0035  0.0082  0.0033 -0.013 ]  after: [ 0.0035  0.0082  0.0033 -0.013 ]


SGD relu: loss 0.6934 -> 0.6934, accuracy 0.47
   first-layer weights before: [ 0.0035  0.0082  0.0033 -0.013 ]  after: [ 0.0035  0.0082  0.0033 -0.013 ]


Adam tanh: loss 0.6934 -> 0.2693, accuracy 0.87
   first-layer weights before: [ 0.0035  0.0082  0.0033 -0.013 ]  after: [ 0.0753 -0.0631 -0.0607 -0.0751]


Adam relu: loss 0.6934 -> 0.6932, accuracy 0.47
   first-layer weights before: [ 0.0035  0.0082  0.0033 -0.013 ]  after: [ 0.0017  0.0094  0.0043 -0.015 ]


SGD tanh, Keras default start: loss 0.5639 -> 0.0021, accuracy 1.00


SGD relu, Keras default start: loss 0.6757 -> 0.0012, accuracy 1.00


## Small and large random weights in a wide network (numpy only)
1000 rows of 500 standard-normal inputs; three hidden layers of 500 nodes; biases 0.
Weights: 0.01 x randn (small) or 1 x randn (large).

In [9]:
def forward(act, scale, layers=3, seed=0):
    rng = np.random.default_rng(seed)
    a = rng.standard_normal((1000, 500))
    out = [a]
    for _ in range(layers):
        W = rng.standard_normal((500, 500)) * scale
        z = a @ W
        a = {"tanh": np.tanh, "sigmoid": lambda v: 1 / (1 + np.exp(-v)), "relu": lambda v: np.maximum(0, v)}[act](z)
        out.append(a)
    return out

hist = []
for act in ("tanh", "sigmoid", "relu"):
    for name, scale in (("small", 0.01), ("large", 1.0)):
        outs = forward(act, scale)
        stats = " | ".join(f"layer {i}: mean {o.mean():8.3f}, std {o.std():8.3f}" for i, o in enumerate(outs[1:], 1))
        print(f"{act:7s} {name}: {stats}")
        if act == "tanh":
            for i, o in enumerate(outs):
                sample = np.random.default_rng(i).choice(o.ravel(), 5000, replace=False).round(4)
                hist += [dict(weights=name, layer=i, value=v) for v in sample]
        if act == "tanh" and name == "large":
            print("   share of tanh outputs beyond +-0.99 in layer 3:", np.mean(np.abs(outs[3]) > 0.99).round(3))
        if act == "tanh" and name == "small":
            print("   share of tanh outputs within +-0.01 in layer 3:", np.mean(np.abs(outs[3]) < 0.01).round(3))
pd.DataFrame(hist).to_csv("data/tanh_hist.csv.gz", index=False)

tanh    small: layer 1: mean    0.000, std    0.214 | layer 2: mean    0.000, std    0.048 | layer 3: mean    0.000, std    0.011
   share of tanh outputs within +-0.01 in layer 3: 0.656
tanh    large: layer 1: mean    0.001, std    0.982 | layer 2: mean    0.000, std    0.982 | layer 3: mean    0.001, std    0.982
   share of tanh outputs beyond +-0.99 in layer 3: 0.904


sigmoid small: layer 1: mean    0.500, std    0.055 | layer 2: mean    0.500, std    0.027 | layer 3: mean    0.500, std    0.027
sigmoid large: layer 1: mean    0.500, std    0.482 | layer 2: mean    0.501, std    0.473 | layer 3: mean    0.503, std    0.474


relu    small: layer 1: mean    0.089, std    0.131 | layer 2: mean    0.014, std    0.021 | layer 3: mean    0.002, std    0.003
relu    large: layer 1: mean    8.947, std   13.081 | layer 2: mean  140.354, std  205.436 | layer 3: mean 2224.035, std 3216.880


## Large weights drawn uniformly between 0 and 1 (tanh, same 500-node network)

In [10]:
rng = np.random.default_rng(0)
a = rng.standard_normal((1000, 500))
for layer in range(1, 4):
    a = np.tanh(a @ rng.uniform(0, 1, (500, 500)))
    print(f"layer {layer}: share of tanh outputs beyond +-0.99: {np.mean(np.abs(a) > 0.99):.3f}")

layer 1: share of tanh outputs beyond +-0.99: 0.839
layer 2: share of tanh outputs beyond +-0.99: 0.993
layer 3: share of tanh outputs beyond +-0.99: 1.000
